In [ ]:
import os
from pathlib import Path 

import tqdm
import yaml
import pandas as pd
import numpy as np

from benchmark_data_leakage.chembl_requester import ChEMBLRequester
from benchmark_data_leakage.utils import find_repo_root, get_canonical_smiles_from_smiles

repo_root = find_repo_root()
data_dir = repo_root / "data"
out_dir = data_dir / "out"
out_dir.mkdir(exist_ok=True, parents=True)

### Extract the FEP+ 4 data

In [ ]:
# NOTE: You need to clone https://github.com/openforcefield/protein-ligand-benchmark/tree/0.2.1
# and add the root to that repo here. v0.2.1 is important!
protein_ligand_benchmark_root =


TARGET_MAPPING_FEP_4 = {
    # Keys are folder names as in https://github.com/openforcefield/protein-ligand-benchmark/tree/0.2.1/data
    # Values are (gene_name, uniprot_id, chembl_target_id)
    # Note that gene_name is from ChEMBL, e.g. MAPK8 -> JNK1 and MAPK14 -> p38 alpha
    "2019-12-09_p38": ("MAPK14", "Q16539", "CHEMBL260"),
    "2019-09-23_jnk1": ("MAPK8", "P45983", "CHEMBL2276"),
    "2020-02-07_tyk2": ("TYK2", "P29597", "CHEMBL3553"),
    "2019-12-13_cdk2": ("CDK2", "P24941", "CHEMBL301"),
}

def measurement_to_pchembl(value: float, unit: str) -> float:
    """Convert a measurement value + unit to pchembl (-log10(M))."""
    unit = unit.lower().strip()
    multipliers = {"nm": 1e-9, "um": 1e-6, "mm": 1e-3, "m": 1.0}
    if unit not in multipliers:
        raise ValueError(f"Unknown measurement unit: {unit!r}")
    return -np.log10(value * multipliers[unit])


def extract_benchmark_data(root_path):
    data_dir = os.path.join(root_path, 'data')
    all_records = []

    # Iterate through each folder in the 'data' directory (each represents a gene)
    if not os.path.exists(data_dir):
        print(f"Error: {data_dir} not found.")
        return pd.DataFrame()

    for folder_name in os.listdir(data_dir):
        gene_path = os.path.join(data_dir, folder_name)

        # Check if it's a directory
        if not os.path.isdir(gene_path):
            continue

        # Target the specific ligands.yml file
        ligands_file = os.path.join(gene_path, '00_data', 'ligands.yml')

        if os.path.exists(ligands_file):
            with open(ligands_file, 'r') as f:
                try:
                    data = yaml.safe_load(f)
                    if not data:
                        continue

                    for lig_id, info in data.items():
                        # Extract measurement details safely
                        meas = info.get('measurement', {})

                        record = {
                            'folder_name': folder_name,
                            'ligand_name': info.get('name'),
                            'smiles': info.get('smiles'),
                            'measurement_value': meas.get('value'),
                            'measurement_unit': meas.get('unit'),
                            'measurement_comment': meas.get('comment'),
                            'measurement_doi': meas.get('doi'),
                            'measurement_type': meas.get('type'),
                            'measurement_error': meas.get('error'),
                        }
                        all_records.append(record)
                except yaml.YAMLError as exc:
                    print(f"Error parsing {ligands_file}: {exc}")

    # Create the pandas DataFrame
    df = pd.DataFrame(all_records)

    # Reorder columns as requested
    cols = ['folder_name', 'ligand_name', 'smiles', 'measurement_comment', 'measurement_doi', 
            'measurement_type', 'measurement_error', 'measurement_unit', 'measurement_value']
    return df[cols]

benchmark_df = extract_benchmark_data(protein_ligand_benchmark_root)
print(f"The full benchmark was extracted data loaded {benchmark_df.shape}")

# Process all data
benchmark_df = benchmark_df.loc[
    benchmark_df["folder_name"].isin(TARGET_MAPPING_FEP_4.keys())
].copy()
benchmark_df["gene_name"] = None
benchmark_df["uniprot_id"] = None
benchmark_df["chembl_target_id"] = None
benchmark_df["assay_id"] = None
for id_, row in benchmark_df.iterrows():
    # Assign target info
    gene_name, uniprot_id, chembl_target_id = TARGET_MAPPING_FEP_4[row["folder_name"]]
    benchmark_df.loc[id_, "gene_name"] = gene_name
    benchmark_df.loc[id_, "uniprot_id"] = uniprot_id
    benchmark_df.loc[id_, "chembl_target_id"] = chembl_target_id
    benchmark_df.loc[id_, "assay_id"] = uniprot_id
    # Canonicalise the smiles
    smiles = row["smiles"]
    canonical_smiles = get_canonical_smiles_from_smiles(smiles)
    benchmark_df.loc[id_, "smiles"] = canonical_smiles
    # Add pchembl_value (e.g. pIC50, pKi, pKd)
    benchmark_df.loc[id_, "pchembl_value"] = measurement_to_pchembl(row["measurement_value"], row["measurement_unit"])
benchmark_df = benchmark_df.rename(columns={"measurement_type": "standard_type"})

print(f"Data has been filtered and prepared {benchmark_df.shape}")

### Add chembl_ligand_id for those that exist in chembl. This takes time!

In [ ]:
# This is required to map the FEP+ datapoints to ChEMBL ligand id
with open(repo_root / "chembl_db.yaml") as f:
    chembl_db = yaml.safe_load(f)
chembl_req = ChEMBLRequester(**chembl_db)

chembl_id_to_smiles = chembl_req.get_chembl_id_to_smiles()

# Canonicalise the smiles so that it's the same format as the FEP data above.
# This could take some time
smiles_to_chembl_id = {}
for chembl_lig_id, smiles in tqdm.tqdm(chembl_id_to_smiles, desc="Canonicalising smiles"):
    try:
        # ChEMBL smiles should already be canonical, doing this anyway as exact implementations differ
        canonical_smiles = get_canonical_smiles_from_smiles(smiles)
    except ValueError:
        print(f"Failed to canonicalise {chembl_lig_id}, skipping")
        continue
    smiles_to_chembl_id[smiles] = chembl_lig_id

benchmark_df["chembl_ligand_id"] = benchmark_df["smiles"].map(smiles_to_chembl_id)
print(f"Fraction with chembl_lig_id: {(~benchmark_df['chembl_ligand_id'].isna()).mean()}")

### Save the data

In [ ]:
out_fp = out_dir / "FEPp_benchmark.csv"
print(f"Dumping data to {out_fp}")
benchmark_df.to_csv(out_fp, index=False)